In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tahap-tahap Transpiler

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami sarankan menggunakan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Halaman ini menjelaskan tahap-tahap dari pipeline transpilasi bawaan di Qiskit SDK. Ada enam tahap:

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

Fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) membuat [staged pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) preset yang terdiri dari tahap-tahap ini. Pass spesifik yang membentuk setiap tahap bergantung pada argumen yang diberikan ke `generate_preset_pass_manager`. `optimization_level` adalah argumen posisional yang wajib ditentukan; nilainya berupa integer yang bisa 0, 1, 2, atau 3. Nilai yang lebih tinggi menunjukkan optimasi yang lebih berat namun lebih mahal (lihat [Pengaturan default dan opsi konfigurasi Transpiler](defaults-and-configuration-options)).

Cara yang disarankan untuk mentranspilasi Circuit adalah dengan membuat preset staged pass manager lalu menjalankannya pada Circuit, seperti yang dijelaskan di [Transpile dengan pass manager](transpile-with-pass-managers). Namun, alternatif yang lebih sederhana tapi kurang bisa dikustomisasi adalah menggunakan fungsi [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Fungsi ini menerima Circuit langsung sebagai argumen. Sama seperti `generate_preset_pass_manager`, pass transpiler yang digunakan bergantung pada argumen seperti `optimization_level` yang diberikan ke `transpile`. Sebenarnya, secara internal fungsi `transpile` memanggil `generate_preset_pass_manager` untuk membuat preset staged pass manager dan menjalankannya pada Circuit.

## Tahap Init
Tahap pertama ini secara default tidak banyak melakukan hal dan terutama berguna jika kamu ingin menyertakan optimasi awal sendiri. Karena kebanyakan algoritma layout dan routing hanya dirancang untuk bekerja dengan Gate satu dan dua Qubit, tahap ini juga digunakan untuk menerjemahkan Gate apa pun yang beroperasi pada lebih dari dua Qubit menjadi Gate yang hanya beroperasi pada satu atau dua Qubit.

Untuk informasi lebih lanjut tentang mengimplementasikan optimasi awal sendiri untuk tahap ini, lihat bagian tentang plugin dan penyesuaian pass manager.

## Tahap Layout
Tahap berikutnya melibatkan layout atau konektivitas Backend tempat Circuit akan dikirim. Secara umum, sirkuit kuantum adalah entitas abstrak yang Qubit-nya merupakan representasi "virtual" atau "logis" dari Qubit aktual yang digunakan dalam komputasi. Untuk mengeksekusi serangkaian Gate, diperlukan pemetaan satu-ke-satu dari Qubit "virtual" ke Qubit "fisik" pada perangkat kuantum nyata. Pemetaan ini disimpan sebagai objek `Layout` dan merupakan bagian dari batasan yang didefinisikan dalam [instruction set architecture (ISA)](./transpile#instruction-set-architecture) Backend.

![Gambar ini mengilustrasikan Qubit yang dipetakan dari representasi wire ke diagram yang menggambarkan cara Qubit terhubung pada QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Pemetaan Qubit")

Pemilihan pemetaan sangat penting untuk meminimalkan jumlah operasi SWAP yang diperlukan untuk memetakan Circuit input ke topologi perangkat dan memastikan Qubit yang paling terkalibrasi digunakan. Karena pentingnya tahap ini, preset pass manager mencoba beberapa metode berbeda untuk menemukan layout terbaik. Biasanya ini melibatkan dua langkah: pertama, coba temukan layout "sempurna" (layout yang tidak memerlukan operasi SWAP apa pun), kemudian, pass heuristik yang mencoba menemukan layout terbaik jika layout sempurna tidak ditemukan. Ada dua `Pass` yang biasanya digunakan untuk langkah pertama ini:

- `TrivialLayout`: Memetakan setiap Qubit virtual secara naif ke Qubit fisik bernomor sama pada perangkat (yaitu, [`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]). Ini adalah perilaku historis yang hanya digunakan di `optimzation_level=1` untuk mencoba menemukan layout sempurna. Jika gagal, `VF2Layout` dicoba berikutnya.
- `VF2Layout`: Ini adalah `AnalysisPass` yang memilih layout ideal dengan memperlakukan tahap ini sebagai masalah subgraph isomorphism, diselesaikan oleh algoritma VF2++. Jika lebih dari satu layout ditemukan, heuristik penilaian dijalankan untuk memilih pemetaan dengan rata-rata error terendah.

Kemudian untuk tahap heuristik, dua pass digunakan secara default:

- `DenseLayout`: Menemukan sub-graph dari perangkat dengan konektivitas terbesar dan memiliki jumlah Qubit yang sama dengan Circuit (digunakan untuk optimization level 1 jika ada operasi control flow seperti IfElseOp dalam Circuit).
- `SabreLayout`: Pass ini memilih layout dengan memulai dari layout acak awal dan berulang kali menjalankan algoritma `SabreSwap`. Pass ini hanya digunakan pada optimization level 1, 2, dan 3 jika layout sempurna tidak ditemukan melalui pass `VF2Layout`. Untuk detail lebih lanjut tentang algoritma ini, lihat makalah [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)

## Tahap Routing
Untuk mengimplementasikan Gate dua Qubit antara Qubit yang tidak terhubung langsung pada perangkat kuantum, satu atau lebih Gate SWAP harus disisipkan ke dalam Circuit untuk memindahkan status Qubit hingga bersebelahan pada peta Gate perangkat. Setiap Gate SWAP mewakili operasi yang mahal dan berisik untuk dilakukan. Dengan demikian, menemukan jumlah minimum Gate SWAP yang diperlukan untuk memetakan Circuit ke perangkat tertentu adalah langkah penting dalam proses transpilasi. Untuk efisiensi, tahap ini biasanya dihitung bersamaan dengan tahap Layout secara default, namun keduanya secara logis berbeda satu sama lain. Tahap *Layout* memilih Qubit hardware yang akan digunakan, sementara tahap *Routing* menyisipkan jumlah Gate SWAP yang tepat agar Circuit dapat dieksekusi menggunakan layout yang dipilih.

Namun, menemukan pemetaan SWAP yang optimal itu sulit. Bahkan, ini adalah masalah NP-hard, sehingga terlalu mahal untuk dihitung kecuali untuk perangkat kuantum dan Circuit input yang sangat kecil. Untuk mengatasi hal ini, Qiskit menggunakan algoritma heuristik stokastik bernama `SabreSwap` untuk menghitung pemetaan SWAP yang baik, meskipun tidak selalu optimal. Penggunaan metode stokastik berarti Circuit yang dihasilkan tidak dijamin sama di setiap kali dijalankan. Memang, menjalankan Circuit yang sama berulang kali menghasilkan distribusi kedalaman Circuit dan jumlah Gate pada output. Itulah mengapa banyak pengguna memilih untuk menjalankan fungsi routing (atau seluruh `StagedPassManager`) berkali-kali dan memilih Circuit dengan kedalaman terendah dari distribusi output.

Misalnya, mari kita ambil Circuit GHZ 15 Qubit yang dieksekusi 100 kali, menggunakan `initial_layout` yang "buruk" (terputus).

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Seperti yang bisa kamu lihat, Circuit ini harus mengeksekusi Gate dua Qubit antara Qubit 0 dan 14, yang sangat jauh satu sama lain pada graf konektivitas. Menjalankan Circuit ini dengan demikian memerlukan penyisipan Gate SWAP untuk mengeksekusi semua Gate dua Qubit menggunakan pass `SabreSwap`.

Perhatikan juga bahwa algoritma `SabreSwap` berbeda dari metode `SabreLayout` yang lebih besar di tahap sebelumnya. Secara default, `SabreLayout` menjalankan layout dan routing, lalu mengembalikan Circuit yang sudah ditransformasi. Hal ini dilakukan karena beberapa alasan teknis tertentu yang disebutkan di [halaman referensi API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) pass tersebut.

## Tahap Translation
Saat menulis Circuit kuantum, kamu bebas menggunakan Gate kuantum (operasi uniter) apa pun yang kamu suka, beserta kumpulan operasi non-Gate seperti instruksi pengukuran atau reset Qubit. Namun, kebanyakan perangkat kuantum hanya mendukung sejumlah kecil Gate kuantum dan operasi non-Gate secara native. Gate native ini merupakan bagian dari definisi [ISA](./transpile#instruction-set-architecture) target, dan tahap preset `PassManagers` ini menerjemahkan (atau *unrolls*) Gate yang ditentukan dalam Circuit ke Gate basis native Backend yang ditentukan. Ini adalah langkah penting, karena memungkinkan Circuit untuk dieksekusi oleh Backend, namun biasanya mengakibatkan peningkatan kedalaman dan jumlah Gate.

Dua kasus khusus sangat penting untuk disorot, dan membantu mengilustrasikan apa yang dilakukan tahap ini.

1. Jika Gate SWAP bukan Gate native untuk Backend target, ini memerlukan tiga Gate CNOT:

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

Sebagai produk dari tiga Gate CNOT, SWAP adalah operasi yang mahal untuk dilakukan pada perangkat kuantum yang berisik. Namun, operasi semacam itu biasanya diperlukan untuk menanamkan Circuit ke dalam konektivitas Gate yang terbatas dari banyak perangkat. Dengan demikian, meminimalkan jumlah Gate SWAP dalam Circuit adalah tujuan utama dalam proses transpilasi.

2. Gate Toffoli, atau controlled-controlled-not (`ccx`), adalah Gate tiga Qubit. Mengingat bahwa set Gate basis kita hanya mencakup Gate satu dan dua Qubit, operasi ini harus didekomposisi. Namun, ini cukup mahal:

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Untuk setiap Gate Toffoli dalam Circuit kuantum, hardware dapat mengeksekusi hingga enam Gate CNOT dan sejumlah Gate satu Qubit. Contoh ini menunjukkan bahwa algoritma apa pun yang menggunakan banyak Gate Toffoli akan berakhir sebagai Circuit dengan kedalaman besar dan karenanya akan terpengaruh secara signifikan oleh noise.

## Tahap Optimization
Tahap ini berpusat pada dekomposisi Circuit kuantum ke dalam set Gate basis perangkat target, dan harus melawan peningkatan kedalaman dari tahap layout dan routing. Untungnya, ada banyak rutinitas untuk mengoptimalkan Circuit dengan menggabungkan atau menghilangkan Gate. Dalam beberapa kasus, metode-metode ini sangat efektif sehingga Circuit output memiliki kedalaman lebih rendah dari input, bahkan setelah layout dan routing ke topologi hardware. Dalam kasus lain, tidak banyak yang bisa dilakukan, dan komputasi mungkin sulit dilakukan pada perangkat yang berisik. Tahap inilah tempat berbagai optimization level mulai berbeda.

- Untuk `optimization_level=1`, tahap ini menyiapkan [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) dan [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), yang menggabungkan rantai Gate satu Qubit dan membatalkan Gate CNOT yang berdampingan.
- Untuk `optimization_level=2`, tahap ini menggunakan pass [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) sebagai pengganti `CXCancellation`, yang menghapus Gate redundan dengan memanfaatkan relasi komutasi.
- Untuk `optimization_level=3`, tahap ini menyiapkan pass-pass berikut:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

Selain itu, tahap ini juga melakukan beberapa pemeriksaan akhir untuk memastikan semua instruksi dalam Circuit terdiri dari Gate basis yang tersedia pada Backend target.

Contoh di bawah menggunakan state GHZ menunjukkan efek dari pengaturan optimization level yang berbeda pada kedalaman Circuit dan jumlah Gate.

> **Note:** Output transpilasi bervariasi karena SWAP mapper yang stokastik. Oleh karena itu, angka-angka di bawah kemungkinan akan berubah setiap kali kamu menjalankan kode.

![State GHZ 15 Qubit](../docs/images/guides/transpiler-stages/transpiler-11.avif "State GHZ 15 Qubit sebelum transpilasi")

Kode berikut membangun state GHZ 15 Qubit dan membandingkan `optimization_levels` transpilasi dari segi kedalaman Circuit hasil, jumlah Gate, dan jumlah Gate multi-Qubit.